In [1]:
import pandas as pd
import mysql.connector
from getpass import getpass

# โหลด dim_park
dim_park = pd.read_csv(
    "../data/processed/dim_park.csv",
    encoding="utf-8-sig"
)

# เปลี่ยน NaN เป็น None เพื่อให้ MySQL เก็บเป็น NULL
dim_park = dim_park.astype(object).where(pd.notna(dim_park), None)

# ให้กรอกรหัส root โดยไม่แสดงรหัสในโค้ด
password = getpass("MySQL root password: ")

# เชื่อมต่อ MySQL
conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=password,
    database="national_park_analytics"
)

cursor = conn.cursor()

sql = """
INSERT INTO dim_park
    (park_name, province, office, latitude, longitude)
VALUES
    (%s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    province = VALUES(province),
    office = VALUES(office),
    latitude = VALUES(latitude),
    longitude = VALUES(longitude)
"""

data = list(
    dim_park[
        ["park_name", "province", "office", "latitude", "longitude"]
    ].itertuples(index=False, name=None)
)

cursor.executemany(sql, data)
conn.commit()

# เช็กจำนวนข้อมูล
cursor.execute("SELECT COUNT(*) FROM dim_park")
total = cursor.fetchone()[0]

print("เชื่อมต่อ MySQL สำเร็จ")
print("จำนวนอุทยานใน dim_park:", total)

cursor.close()
conn.close()

เชื่อมต่อ MySQL สำเร็จ
จำนวนอุทยานใน dim_park: 165


In [2]:
# โหลด master dataset
master = pd.read_csv(
    "../data/processed/national_park_master.csv",
    encoding="utf-8-sig"
)

master["date"] = pd.to_datetime(master["date"])

# สร้าง dim_date จากวันที่ที่มีจริงในข้อมูล
dim_date = (
    master[
        ["date", "month_num", "month_th", "calendar_year_be", "calendar_year_ce"]
    ]
    .drop_duplicates()
    .sort_values("date")
    .reset_index(drop=True)
)

dim_date["date_id"] = range(1, len(dim_date) + 1)

dim_date = dim_date[
    [
        "date_id",
        "date",
        "month_num",
        "month_th",
        "calendar_year_be",
        "calendar_year_ce"
    ]
]

print(dim_date.shape)
display(dim_date.head())

(93, 6)


,date_id,date,month_num,month_th,calendar_year_be,calendar_year_ce
0,1,2018-01-01,1,ม.ค.,2561,2018
1,2,2018-02-01,2,ก.พ.,2561,2018
2,3,2018-03-01,3,มี.ค.,2561,2018
3,4,2018-04-01,4,เม.ย.,2561,2018
4,5,2018-05-01,5,พ.ค.,2561,2018


In [3]:
import mysql.connector
from getpass import getpass

password = getpass("MySQL root password: ")

conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=password,
    database="national_park_analytics"
)

cursor = conn.cursor()

sql = """
INSERT INTO dim_date (
    date_id,
    date,
    month_num,
    month_th,
    calendar_year_be,
    calendar_year_ce
)
VALUES (%s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    month_num = VALUES(month_num),
    month_th = VALUES(month_th),
    calendar_year_be = VALUES(calendar_year_be),
    calendar_year_ce = VALUES(calendar_year_ce)
"""

data = [
    (
        row.date_id,
        row.date.date(),
        row.month_num,
        row.month_th,
        row.calendar_year_be,
        row.calendar_year_ce
    )
    for row in dim_date.itertuples(index=False)
]

cursor.executemany(sql, data)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM dim_date")
total = cursor.fetchone()[0]

print("จำนวนข้อมูลใน dim_date:", total)

cursor.close()
conn.close()

จำนวนข้อมูลใน dim_date: 93


In [4]:
import pandas as pd
import numpy as np
import mysql.connector
from getpass import getpass

# โหลด Master Dataset
master = pd.read_csv(
    "../data/processed/national_park_master.csv",
    encoding="utf-8-sig"
)

master["date"] = pd.to_datetime(master["date"]).dt.date

password = getpass("MySQL root password: ")

conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=password,
    database="national_park_analytics"
)

cursor = conn.cursor()

# -------------------------
# ดึง park_id จาก dim_park
# -------------------------
cursor.execute("""
    SELECT park_id, park_name
    FROM dim_park
""")

park_lookup = pd.DataFrame(
    cursor.fetchall(),
    columns=["park_id", "park_name"]
)

# -------------------------
# ดึง date_id จาก dim_date
# -------------------------
cursor.execute("""
    SELECT date_id, date
    FROM dim_date
""")

date_lookup = pd.DataFrame(
    cursor.fetchall(),
    columns=["date_id", "date"]
)

date_lookup["date"] = pd.to_datetime(date_lookup["date"]).dt.date

# -------------------------
# Map Dimension IDs
# -------------------------
fact = master.merge(
    park_lookup,
    on="park_name",
    how="left"
)

fact = fact.merge(
    date_lookup,
    on="date",
    how="left"
)

print("Missing park_id:", fact["park_id"].isna().sum())
print("Missing date_id:", fact["date_id"].isna().sum())

assert fact["park_id"].isna().sum() == 0
assert fact["date_id"].isna().sum() == 0

Missing park_id: 0
Missing date_id: 0


In [5]:
fact_load = fact[
    [
        "park_id",
        "date_id",
        "fiscal_year_be",
        "visitors",
        "min_rain",
        "max_rain",
        "avg_rain",
        "total_attractions",
        "open_attractions",
        "closed_attractions",
        "unknown_attractions",
        "known_attractions",
        "availability_pct",
        "unknown_pct",
        "availability_data_status"
    ]
].copy()


# แปลง NaN → NULL สำหรับ MySQL
def clean_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


data = [
    tuple(clean_value(value) for value in row)
    for row in fact_load.itertuples(index=False, name=None)
]


# Full Refresh เพื่อให้รัน Notebook ซ้ำได้โดยข้อมูลไม่ซ้ำ
cursor.execute("TRUNCATE TABLE fact_park_monthly")


insert_sql = """
INSERT INTO fact_park_monthly (
    park_id,
    date_id,
    fiscal_year_be,
    visitors,
    min_rain,
    max_rain,
    avg_rain,
    total_attractions,
    open_attractions,
    closed_attractions,
    unknown_attractions,
    known_attractions,
    availability_pct,
    unknown_pct,
    availability_data_status
)
VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s
)
"""

cursor.executemany(insert_sql, data)

conn.commit()


# เช็กจำนวนข้อมูล
cursor.execute("""
    SELECT COUNT(*)
    FROM fact_park_monthly
""")

total = cursor.fetchone()[0]

print("โหลด Fact Table สำเร็จ")
print("จำนวนแถว:", total)

cursor.close()
conn.close()

โหลด Fact Table สำเร็จ
จำนวนแถว: 14427
